# KAMP 사출성형 데이터 전처리

오늘의 목표는 EDA에서 확인한 문제를 바탕으로 **모델링 가능한 데이터셋**을 만드는 것입니다.

대상 데이터:
- `train_cn7`
- `train_rg3`

Target:
- `PassOrFail`

전처리 핵심 원칙:
1. 공정적으로 불가능한 값과 통계적 이상치를 구분한다.
2. 불량 신호가 될 수 있는 이상치를 무조건 제거하지 않는다.
3. Train / Validation 분리 전에 데이터 누수가 발생하지 않도록 한다.
4. 원본 데이터와 전처리 완료 데이터를 분리하여 관리한다.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(
    r"C:\dev\KAMP_hoochuteam\data\train\moldset_labeled_cn7.csv"
)
train_rg3 = pd.read_csv(
    r"C:\dev\KAMP_hoochuteam\data\train\moldset_labeled_rg3.csv"
)

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

TARGET = "PassOrFail"

DATASETS = {
    "CN7": train_cn7.copy(),
    "RG3": train_rg3.copy()
}

## 1. 데이터 복사 및 기본 구조 확인

### 문제
원본 데이터를 보존하기 위해 `train_cn7`, `train_rg3`를 복사하여 새로운 데이터프레임을 만드세요.

각 데이터셋에 대해 다음을 확인하세요.

- shape
- 컬럼명
- 데이터 타입
- Target 분포

In [5]:
for name, df in DATASETS.items():
    print(f"\n[{name}]")
    print("shape:", df.shape)
    print("\n컬럼:")
    print(df.columns.tolist())
    print("\n데이터 타입:")
    display(df.dtypes.to_frame("dtype"))
    print("\nTarget 분포:")
    display(df[TARGET].value_counts(dropna=False).to_frame("count"))


[CN7]
shape: (1211, 26)

컬럼:
['Unnamed: 0', 'PassOrFail', 'Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']

데이터 타입:


,dtype
Unnamed: 0,int64
PassOrFail,int64
Injection_Time,float64
Filling_Time,float64
Plasticizing_Time,float64
Cycle_Time,float64
Clamp_Close_Time,float64
Cushion_Position,float64
Plasticizing_Position,float64
Clamp_Open_Position,float64



Target 분포:


,count
PassOrFail,
0,1194
1,17



[RG3]
shape: (1182, 26)

컬럼:
['Unnamed: 0', 'PassOrFail', 'Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']

데이터 타입:


,dtype
Unnamed: 0,int64
PassOrFail,int64
Injection_Time,float64
Filling_Time,float64
Plasticizing_Time,float64
Cycle_Time,float64
Clamp_Close_Time,float64
Cushion_Position,float64
Plasticizing_Position,float64
Clamp_Open_Position,float64



Target 분포:


,count
PassOrFail,
0,1157
1,25


## 2. 불필요 컬럼 후보 확인

### 문제
각 데이터셋에서 다음 컬럼을 찾아보세요.

- 값이 하나뿐인 컬럼
- 모든 값이 서로 다른 ID성 컬럼 후보
- 분석에 필요하지 않은 index성 컬럼

단, 자동으로 제거하지 말고 **후보만 확인**하세요.

In [7]:
for name, df in DATASETS.items():
    print(f"\n[{name}]")

    nunique = df.nunique(dropna=False)

    constant_cols = nunique[nunique <= 1].index.tolist()
    id_like_cols = nunique[nunique == len(df)].index.tolist()

    print("값이 하나뿐인 컬럼:", constant_cols)
    print("ID성 후보 컬럼:", id_like_cols)


[CN7]
값이 하나뿐인 컬럼: ['Clamp_Open_Position']
ID성 후보 컬럼: ['Unnamed: 0']

[RG3]
값이 하나뿐인 컬럼: ['Clamp_Open_Position']
ID성 후보 컬럼: ['Unnamed: 0']


## 3. 결측치 확인

### 문제
각 컬럼별 결측치 개수와 비율을 확인하세요.

결측치가 존재한다면 다음 기준으로 처리 방법을 결정하세요.

- 수치형 변수: 중앙값 대체 후보
- 범주형 변수: 최빈값 또는 `Unknown`
- 센서 변수: 단순 대체 전에 공정 이상 여부 확인

In [8]:
for name, df in DATASETS.items():
    missing = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_ratio": df.isnull().mean() * 100
    }).sort_values("missing_ratio", ascending=False)

    print(f"\n[{name}] 결측치")
    display(missing[missing["missing_count"] > 0])


[CN7] 결측치


,missing_count,missing_ratio



[RG3] 결측치


,missing_count,missing_ratio


In [ ]:
# 기본적인 결측치 처리 예시
for name, df in DATASETS.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")
    categorical_cols = df.select_dtypes(exclude=np.number).columns

    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    for col in categorical_cols:
        if df[col].isnull().any():
            mode = df[col].mode()
            df[col] = df[col].fillna(mode.iloc[0] if len(mode) else "Unknown")

    DATASETS[name] = df

## 4. 완전 중복 데이터 확인

### 문제
각 데이터셋에서 완전히 동일한 행의 개수를 확인하세요.

완전 중복 행이 발견되더라도 제조 공정에서 반복 생산된 데이터일 수 있으므로,
바로 삭제하지 말고 중복 여부만 확인하세요.

In [ ]:
for name, df in DATASETS.items():
    print(f"[{name}] 완전 중복 행:", df.duplicated().sum())

## 5. 동일 X + 동일 Target 확인

### 문제
Target을 제외한 모든 설명변수가 동일하면서 `PassOrFail` 값도 동일한 데이터가 얼마나 있는지 확인하세요.

이 데이터는 반복 생산 데이터일 가능성이 있습니다.

In [ ]:
for name, df in DATASETS.items():
    feature_cols = [c for c in df.columns if c != TARGET]

    same_x_same_y = df.duplicated(subset=feature_cols + [TARGET], keep=False)

    print(f"[{name}] 동일 X + 동일 Target 행 수:", same_x_same_y.sum())

## 6. 동일 X + 다른 Target 확인

### 문제
설명변수는 완전히 동일하지만 `PassOrFail` 값이 다른 데이터가 존재하는지 확인하세요.

이 경우 다음 가능성을 고려할 수 있습니다.

- 미관측 변수 존재
- 센서 측정 한계
- 공정 노이즈
- 라벨 노이즈

In [ ]:
for name, df in DATASETS.items():
    feature_cols = [c for c in df.columns if c != TARGET]

    target_nunique = (
        df.groupby(feature_cols, dropna=False)[TARGET]
          .nunique()
    )

    conflict_count = (target_nunique > 1).sum()

    print(f"[{name}] 동일 X + 다른 Target 조합 수:", conflict_count)

## 7. 공정적으로 비정상적인 값 확인

### 문제
시간, 속도, RPM 등 물리적으로 음수가 나오기 어려운 수치형 변수에서 음수 또는 0 이하 값이 존재하는지 확인하세요.

예시:
- `Injection_Time`
- `Filling_Time`
- `Plasticizing_Time`
- `Cycle_Time`
- `Max_Injection_Speed`
- `Max_Screw_RPM`
- `Average_Screw_RPM`

※ 실제 컬럼 존재 여부를 확인한 뒤 검사하세요.

In [ ]:
CHECK_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM"
]

for name, df in DATASETS.items():
    print(f"\n[{name}]")

    for col in CHECK_COLS:
        if col in df.columns:
            print(
                col,
                "| 음수:", (df[col] < 0).sum(),
                "| 0 이하:", (df[col] <= 0).sum()
            )

## 8. IQR 기반 이상치 탐색

### 문제
수치형 변수에 대해 IQR 방식으로 이상치 개수를 계산하세요.

단, 이 단계에서는 이상치를 제거하지 않습니다.

In [ ]:
outlier_summary = {}

for name, df in DATASETS.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")
    rows = []

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (df[col] < lower) | (df[col] > upper)

        rows.append({
            "column": col,
            "outlier_count": mask.sum(),
            "outlier_ratio": mask.mean() * 100
        })

    outlier_summary[name] = pd.DataFrame(rows).sort_values(
        "outlier_ratio", ascending=False
    )

    print(f"\n[{name}]")
    display(outlier_summary[name])

## 9. 이상치와 Target 관계 확인

### 문제
각 변수의 IQR 이상치 데이터에서 `PassOrFail` 분포를 확인하세요.

특히 불량 데이터가 이상치 영역에 집중되는 변수가 있는지 확인하세요.

> 불량 데이터가 이상치에 집중된다면 해당 값은 제거 대상이 아니라 중요한 불량 신호일 수 있습니다.

In [ ]:
for name, df in DATASETS.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")

    print(f"\n========== {name} ==========")

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (df[col] < lower) | (df[col] > upper)

        if mask.sum() > 0:
            print(f"\n[{col}] 이상치 Target 분포")
            display(df.loc[mask, TARGET].value_counts(normalize=True).to_frame("ratio"))

## 10. Train / Validation 분리

### 문제
`PassOrFail`을 Target으로 하여 Train / Validation 데이터를 분리하세요.

조건:
- `test_size=0.2`
- `random_state=42`
- 분류 문제이므로 Target 비율을 유지하도록 stratify 사용

CN7과 RG3를 각각 분리하세요.

In [ ]:
SPLIT_DATA = {}

for name, df in DATASETS.items():
    X = df.drop(columns=TARGET)
    y = df[TARGET]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    SPLIT_DATA[name] = {
        "X_train": X_train,
        "X_valid": X_valid,
        "y_train": y_train,
        "y_valid": y_valid
    }

    print(
        f"{name} | "
        f"X_train: {X_train.shape} | "
        f"X_valid: {X_valid.shape}"
    )

## 11. Scaling 대상 구분

### 문제
수치형 변수 목록을 만든 뒤 `RobustScaler`를 사용하여 스케일링하세요.

주의:
- Scaler는 Train 데이터에만 `fit`
- Validation에는 `transform`만 수행

Tree 계열 모델은 원본 데이터도 함께 사용할 수 있도록 별도 데이터셋을 유지하세요.

In [ ]:
SCALED_DATA = {}

for name, data in SPLIT_DATA.items():
    X_train = data["X_train"].copy()
    X_valid = data["X_valid"].copy()

    numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

    scaler = RobustScaler()

    X_train_scaled = X_train.copy()
    X_valid_scaled = X_valid.copy()

    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_valid_scaled[numeric_cols] = scaler.transform(X_valid[numeric_cols])

    SCALED_DATA[name] = {
        "X_train_scaled": X_train_scaled,
        "X_valid_scaled": X_valid_scaled,
        "scaler": scaler,
        "numeric_cols": numeric_cols
    }

    print(f"{name} 스케일링 완료 | 수치형 변수 {len(numeric_cols)}개")

## 12. 공정 관계형 파생변수 생성

### 문제
존재하는 컬럼만 사용하여 다음과 같은 파생변수를 만들어보세요.

예시:
- `Injection_Time / Cycle_Time`
- `Filling_Time / Cycle_Time`
- `Plasticizing_Time / Cycle_Time`
- `Max_Injection_Speed / Injection_Time`
- `Max_Screw_RPM - Average_Screw_RPM`
- `Plasticizing_Position - Cushion_Position`
- `Clamp_Open_Position - Cushion_Position`

0으로 나누는 경우가 발생하지 않도록 처리하세요.

In [ ]:
def add_process_features(df):
    df = df.copy()
    created = []

    def safe_ratio(new_col, numerator, denominator):
        if numerator in df.columns and denominator in df.columns:
            denom = df[denominator].replace(0, np.nan)
            df[new_col] = df[numerator] / denom
            created.append(new_col)

    safe_ratio("Injection_Cycle_Ratio", "Injection_Time", "Cycle_Time")
    safe_ratio("Filling_Cycle_Ratio", "Filling_Time", "Cycle_Time")
    safe_ratio("Plasticizing_Cycle_Ratio", "Plasticizing_Time", "Cycle_Time")
    safe_ratio("InjectionSpeed_Time_Ratio", "Max_Injection_Speed", "Injection_Time")

    if {"Max_Screw_RPM", "Average_Screw_RPM"}.issubset(df.columns):
        df["Screw_RPM_Gap"] = df["Max_Screw_RPM"] - df["Average_Screw_RPM"]
        created.append("Screw_RPM_Gap")

    if {"Plasticizing_Position", "Cushion_Position"}.issubset(df.columns):
        df["Plasticizing_Cushion_Gap"] = (
            df["Plasticizing_Position"] - df["Cushion_Position"]
        )
        created.append("Plasticizing_Cushion_Gap")

    if {"Clamp_Open_Position", "Cushion_Position"}.issubset(df.columns):
        df["ClampOpen_Cushion_Gap"] = (
            df["Clamp_Open_Position"] - df["Cushion_Position"]
        )
        created.append("ClampOpen_Cushion_Gap")

    return df, created


FEATURED_DATASETS = {}
CREATED_FEATURES = {}

for name, df in DATASETS.items():
    featured_df, created = add_process_features(df)

    FEATURED_DATASETS[name] = featured_df
    CREATED_FEATURES[name] = created

    print(f"{name} 생성 파생변수:", created)

## 13. 높은 상관관계 변수 확인

### 문제
설명변수끼리의 절대 상관계수가 높은 변수 조합을 확인하세요.

기준 예시:
- `|corr| >= 0.95`

이 단계에서도 자동 제거하지 말고 후보만 확인하세요.

In [ ]:
for name, df in FEATURED_DATASETS.items():
    numeric_df = df.select_dtypes(include=np.number).drop(columns=TARGET, errors="ignore")

    corr = numeric_df.corr().abs()

    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    high_corr = (
        upper.stack()
             .reset_index()
             .rename(columns={
                 "level_0": "feature_1",
                 "level_1": "feature_2",
                 0: "corr"
             })
    )

    high_corr = high_corr[high_corr["corr"] >= 0.95].sort_values(
        "corr", ascending=False
    )

    print(f"\n[{name}] | 상관계수 0.95 이상")
    display(high_corr)

## 14. 클래스 불균형 확인

### 문제
Train 데이터에서 `PassOrFail`의 개수와 비율을 확인하세요.

불균형이 존재하더라도 이 단계에서 SMOTE를 바로 적용하지 마세요.

이후 모델링에서 다음 방법을 비교할 수 있습니다.

- `class_weight`
- threshold 조정
- XGBoost / LightGBM / CatBoost 가중치
- SMOTE

In [ ]:
for name, data in SPLIT_DATA.items():
    y_train = data["y_train"]

    result = pd.DataFrame({
        "count": y_train.value_counts(),
        "ratio": y_train.value_counts(normalize=True)
    })

    print(f"\n[{name}] Train Target 분포")
    display(result)

## 15. 전처리 결과 요약

### 문제
CN7과 RG3 각각에 대해 다음 내용을 표 형태로 정리하세요.

- 원본 행 수
- 최종 행 수
- 원본 컬럼 수
- 최종 컬럼 수
- 결측치 수
- Target 0/1 개수
- 생성한 파생변수 수

In [ ]:
summary_rows = []

for name in DATASETS:
    original = DATASETS[name]
    final_df = FEATURED_DATASETS[name]

    summary_rows.append({
        "dataset": name,
        "original_rows": len(original),
        "final_rows": len(final_df),
        "original_cols": original.shape[1],
        "final_cols": final_df.shape[1],
        "missing_count": int(final_df.isnull().sum().sum()),
        "target_0": int((final_df[TARGET] == 0).sum()),
        "target_1": int((final_df[TARGET] == 1).sum()),
        "created_features": len(CREATED_FEATURES[name])
    })

preprocessing_summary = pd.DataFrame(summary_rows)
display(preprocessing_summary)

## 16. 전처리 데이터 저장

### 문제
전처리가 완료된 CN7, RG3 데이터를 각각 CSV로 저장하세요.

권장 파일명:
- `cn7_processed.csv`
- `rg3_processed.csv`

원본 파일을 덮어쓰지 않도록 주의하세요.

In [ ]:
FEATURED_DATASETS["CN7"].to_csv("cn7_processed.csv", index=False)
FEATURED_DATASETS["RG3"].to_csv("rg3_processed.csv", index=False)

print("저장 완료:")
print("- cn7_processed.csv")
print("- rg3_processed.csv")

# 전처리 완료 후 체크

전처리가 끝나면 다음 단계로 넘어갑니다.

1. Baseline 모델 성능 측정
2. 파생변수 적용 전/후 성능 비교
3. CN7 / RG3 개별 모델 비교
4. 클래스 불균형 대응 방법 비교
5. Feature Importance / SHAP 분석
6. 공정 조건과 불량 발생 원인 연결